# QuantSupport Python bindings — a guided tour

This notebook walks through every component of the `quantsupport` Python API:

1. **Dates, periods and calendars** — the time toolkit
2. **Enums** — currencies, market indices, conventions
3. **Market data** — quote stores, curve configurations, FX
4. **The pricing context** — bootstrapping and lifecycle
5. **Exploring bootstrapped curves** — nodes, discount factors, forward rates, pillars
6. **Pricing trades** — swaps and cross-currency swaps with AD sensitivities
7. **XVA** — per-client CSA terms, netting sets, Monte Carlo exposures

Build and install the bindings first (from the repo root):

```bash
pip install maturin
maturin develop --release -m bindings/python/Cargo.toml
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import quantsupport as qs

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

DATA = next(
    path / "examples" / "cva" / "data"
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "examples" / "cva" / "data" / "quotes.json").is_file()
)
DATA

## 1. Dates, periods and calendars

`qs.Date` is an immutable calendar date. It supports arithmetic with integers
(days), `qs.Period` objects, or period strings like `"6M"` / `"1Y6M"`.
Subtracting two dates gives the number of days between them.

In [ ]:
d = qs.Date(2025, 11, 11)          # or qs.Date.parse("2025-11-11")
print(d, "is a", d.weekday())
print("+30 days:      ", d + 30)
print("+6 months:     ", d + "6M")
print("+1 year 6 mo:  ", d + qs.Period.parse("1Y6M"))
print("end of month:  ", d.end_of_month())
print("days to +6M:   ", (d + "6M") - d)

In [ ]:
p = qs.Period(18, qs.TimeUnit.Months)
print(p, "->", p.length, p.units)
print("from frequency:", qs.Period.from_frequency(qs.Frequency.Quarterly))

`qs.Calendar` answers business-day questions: adjusting dates with a
`BusinessDayConvention`, advancing by periods while skipping holidays, and
listing holidays or business days. `qs.DayCounter` measures year fractions
between dates under a market convention.

In [ ]:
cal = qs.Calendar("UnitedStates")   # also: TARGET, Brazil, Chile, WeekendsOnly, NullCalendar
thanksgiving = qs.Date(2025, 11, 27)
print("business day?          ", cal.is_business_day(thanksgiving))
print("adjusted (Following):  ", cal.adjust(thanksgiving, qs.BusinessDayConvention.Following))
print("advance 5 business d.: ", cal.advance(d, "5D"))
print("business days in Nov:  ", cal.business_days_between(qs.Date(2025, 11, 1), qs.Date(2025, 11, 30)))
print("holidays in Nov 2025:  ", cal.holiday_list(qs.Date(2025, 11, 1), qs.Date(2025, 11, 30)))

dc = qs.DayCounter.Actual360
print("Act/360 year fraction: ", dc.year_fraction(d, d + "6M"))

## 2. Enums

All conventions are proper Python enum classes (hashable, comparable, usable as
dict keys). Every API that takes an enum also accepts its string name, and each
enum has a case-insensitive `parse` static method.

`qs.Currency` carries full ISO 4217 metadata:

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        "code": c.code,
        "name": c.name,
        "symbol": c.symbol,
        "precision": c.precision,
        "numeric_code": c.numeric_code,
    }
    for c in [qs.Currency.USD, qs.Currency.EUR, qs.Currency.CLP, qs.Currency.JPY, qs.Currency.BRL]
)

`qs.MarketIndex` identifies every curve, surface and simulation: overnight
rates (`SOFR`, `ESTR`, `ICP`, ...), term rates (`EURIBOR6m`, `TermSOFR3m`, ...),
plus parameterised indices built with static factories.

In [ ]:
print(qs.MarketIndex.SOFR, "|", qs.MarketIndex.EURIBOR6m, "|", qs.MarketIndex.ICP)
print("equity index:", qs.MarketIndex.equity("SPX"))
print("fx pair:     ", qs.MarketIndex.fx_pair("CLP", "USD"))
print("collateral:  ", qs.MarketIndex.collateral("CLP", "USD"))
print("parse:       ", qs.MarketIndex.parse("sofr"))

print(qs.Side.LongReceive.sign(), qs.Side.parse("pay"))
print(qs.Compounding.Compounded, qs.Frequency.Semiannual, qs.Request.Sensitivities)

## 3. Market data

A `qs.QuoteStore` holds all market quotes as of a single reference date. Python reads the JSON files independently with `json.load`; `from_dict` then constructs each market object from native dictionaries and lists. The same data can instead be defined inline without any file-loading API in the bindings.

In [ ]:
with (DATA / "quotes.json").open(encoding="utf-8") as stream:
    quote_data = json.load(stream)
quotes = qs.QuoteStore.from_dict(quote_data)
print("reference date:", quotes.reference_date, "| quotes:", len(quotes.identifiers()))
quotes.to_dataframe().head(8)

`qs.CurveConfiguration` describes how each curve is bootstrapped: which market
index it belongs to, its day counter and interpolator, and which quotes pin its
pillars. `to_dict()` exposes the full configuration for inspection.

In [ ]:
with (DATA / "curve_specs.json").open(encoding="utf-8") as stream:
    curve_data = json.load(stream)
curves = [
    qs.CurveConfiguration.from_dict(spec)
    for spec in curve_data["curve_specs"]
    if spec["market_index"] != "ESTR"
]
for curve in curves:
    cfg = curve.to_dict()
    print(f"{cfg['market_index']!s:<22} {cfg['day_counter']:<10} "
          f"{cfg['interpolator']:<18} {len(cfg['quotes'])} quotes")

with (DATA / "vol_specs.json").open(encoding="utf-8") as stream:
    volatility_data = json.load(stream)
volatility_surfaces = [
    qs.VolatilitySurfaceConfiguration.from_dict(spec)
    for spec in volatility_data["volatility_surfaces"]
]
volatility_cubes = [
    qs.VolatilityCubeConfiguration.from_dict(spec)
    for spec in volatility_data["volatility_cubes"]
]

In [ ]:
fx = qs.FxStore.from_dict([
    {"base": qs.Currency.CLP, "quote": qs.Currency.USD, "rate": 1.0 / 900.0},
])
discounting = qs.DiscountingConfig(currency=qs.Currency.USD, index=qs.MarketIndex.SOFR)
discounting

## 4. The pricing context

`qs.PricingContext` bundles all market data. **Initializing** it bootstraps
every configured curve, builds volatility surfaces/cubes and runs any
configured simulations.

There are two ways to drive its lifecycle:

- `ctx.initialize()` — bootstrap once, then explore the constructed objects
  freely across notebook cells (used in section 5);
- `with ctx: ...` — additionally starts the automatic-differentiation tape,
  which is required for pricing with sensitivities, and releases everything on
  exit (used in sections 6–7).

In [ ]:
ctx = qs.PricingContext(
    quotes=quotes, curves=curves, fx=fx, discounting=discounting,
    volatility_surfaces=volatility_surfaces, volatility_cubes=volatility_cubes,
)
ctx.initialize()

ref = ctx.reference_date
print("reference date:", ref)
print("original curve configurations:", [str(curve.to_dict()["market_index"]) for curve in ctx.curve_configurations])
print("original quotes:", len(ctx.quotes.identifiers()))
print("volatility surfaces:", [str(surface.market_index) for surface in ctx.volatility_surfaces()])
print("volatility cubes:", [str(cube.market_index) for cube in ctx.volatility_cubes()])

## 5. Exploring bootstrapped curves

After initialization the context holds the *constructed* market data. Each
bootstrapped curve is exposed as a `qs.DiscountCurve` view with:

- `nodes()` — the raw date/discount-factor nodes as a DataFrame,
- `discount_factor(date)` and `forward_rate(start, end, compounding, frequency)`,
- `pillars()` — the quote pillars the bootstrap was calibrated to.

(The same pattern applies to `ctx.volatility_surfaces()`, `ctx.volatility_cubes()`
and `ctx.simulations()` when those are configured.)

In [ ]:
for curve in ctx.curves():
    print(f"{curve.market_index!s:<22} ref={curve.reference_date}  "
          f"day_counter={curve.day_counter}  nodes={len(curve.nodes())}")

In [ ]:
sofr = ctx.curve(qs.MarketIndex.SOFR)
sofr.nodes().head(6)

In [ ]:
# Discount-factor curves make the relative shape of each bootstrapped market visible.
fig, ax = plt.subplots(figsize=(9, 4.5))
for curve in ctx.curves():
    nodes = curve.nodes().assign(date=lambda frame: pd.to_datetime(frame["date"]))
    ax.plot(nodes["date"], nodes["discount_factor"], marker="o", ms=3, label=str(curve.market_index))
ax.set(title="Bootstrapped discount curves", xlabel="Maturity", ylabel="Discount factor")
ax.legend(title="Market index", ncols=2)
fig.autofmt_xdate()
plt.show()

In [ ]:
# Discount factors and forward rates at arbitrary dates.
term = pd.DataFrame(
    {
        "tenor": t,
        "discount_factor": sofr.discount_factor(ref + t),
        "fwd_1y": sofr.forward_rate(ref + t, (ref + t) + "1Y",
                                    qs.Compounding.Simple, qs.Frequency.Annual),
    }
    for t in ["1Y", "2Y", "5Y", "10Y", "20Y"]
)
term

In [ ]:
# The quote pillars the curve was calibrated to (bootstrap inputs).
sofr.pillars()

## 6. Pricing trades

Trade dictionaries may use Python dates and enums directly, as shown below. The same constructors also accept strings loaded independently from JSON files. `ctx.evaluate` takes `qs.Request` values (or equivalent names); sensitivities are computed by automatic differentiation with respect to the original market quotes, inside a `with ctx:` block that starts the AD tape.

In [ ]:
swap_data = {
    "identifier": "USD-RATES-IRS-5Y",
    "start_date": ref,
    "maturity_date": ref + "5Y",
    "notional": 10_000_000.0,
    "fixed_rate": 0.0378,
    "currency": qs.Currency.USD,
    "market_index": qs.MarketIndex.SOFR,
    "side": qs.Side.LongReceive,
    "fixed_leg_frequency": qs.Frequency.Quarterly,
    "floating_leg_frequency": qs.Frequency.Semiannual,
}
swap = qs.Swap.from_dict(swap_data)
print(swap.identifier, "|", swap.currency, swap.market_index, swap.side,
      "|", swap.start_date, "->", swap.maturity_date)

with ctx:
    res = ctx.evaluate(swap, [qs.Request.Value, qs.Request.Cashflows, qs.Request.Sensitivities])
    cashflows = res.cashflows
    sens = res.sensitivities

print(f"NPV = {res.price:,.2f}")
cashflows.head(6)

In [ ]:
# Aggregate by date and leg so the payment schedule is readable at a glance.
cashflow_plot = cashflows.assign(payment_date=pd.to_datetime(cashflows["payment_date"]))
cashflow_plot = cashflow_plot.pivot_table(index="payment_date", columns="leg_index", values="amount", aggfunc="sum").fillna(0.0)
ax = cashflow_plot.plot(kind="bar", stacked=True, figsize=(10, 4.5), width=0.8)
ax.set(title=f"{swap.identifier}: scheduled cashflows", xlabel="Payment date", ylabel="Cashflow amount")
ax.legend(title="Leg")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Quote-level sensitivities (per unit move of each input quote), largest first.
sens.reindex(sens.value.abs().sort_values(ascending=False).index).head(8)

In [ ]:
# The largest quote deltas show which market inputs drive the swap value.
top_sens = sens.assign(abs_value=sens["value"].abs()).nlargest(10, "abs_value").sort_values("value")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(top_sens["pillar"], top_sens["value"], color=top_sens["value"].map(lambda value: "#2a9d8f" if value >= 0 else "#e76f51"))
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set(title=f"{swap.identifier}: largest quote sensitivities", xlabel="Value change per unit quote move", ylabel="Quote pillar")
plt.tight_layout()
plt.show()

In [ ]:
xccy_data = {
    "identifier": "USD-CLP-TREASURY-XCCY-5Y",
    "start_date": str(ref),
    "maturity_date": str(ref + "5Y"),
    "domestic_notional": 10_000_000.0,
    "foreign_notional": 10_000_000.0 * 900.0,
    "domestic_currency": "USD",
    "foreign_currency": "CLP",
    "domestic_market_index": "SOFR",
    "foreign_market_index": "ICP",
    "foreign_spread": 0.002,
    "side": "LongReceive",
}
xccy = qs.CrossCurrencySwap(**xccy_data)

with ctx:
    xres = ctx.evaluate(xccy, ["Value"])
print(f"{xccy.identifier} NPV = {xres.price:,.2f}")

## 7. XVA - CSA terms and Monte Carlo exposures

The XVA engine simulates portfolio values along Monte Carlo paths and aggregates CVA/FVA per **netting set**. Each institutional portfolio carries its own `qs.CsaTerms`:

- `collateral_index` / `collateral_currency`: collateral remuneration curve and currency.
- `credit_spread`, `recovery`: credit parameters for CVA.
- `funding_spread`: funding parameters for FVA.
- Optional `credit_index`: a bootstrapped credit curve instead of a flat spread.

`qs.XvaConfig` holds the paths, seed and model configurations. The shipped configuration calibrates SOFR to the constructed caplet surface and ICP to the swaption cube loaded in section 3; its collateral curve shares the ICP driver.

In [ ]:
with (DATA / "xva_config.json").open(encoding="utf-8") as stream:
    xva_configuration_data = json.load(stream)
with (DATA / "csa_terms.json").open(encoding="utf-8") as stream:
    csa_data = json.load(stream)
xva_config = qs.XvaConfig.from_dict(xva_configuration_data)
csa = qs.CsaTerms.from_dict(csa_data)
print(xva_config)
print(csa)

csa_rates_desk = qs.CsaTerms.from_dict({
    "collateral_index": "SOFR",
    "collateral_currency": "USD",
    "credit_spread": 0.02,
    "recovery": 0.4,
    "funding_spread": 0.005,
})

In [ ]:
with ctx:
    result = ctx.run_xva(
        xva_config,
        netting_sets=[
            qs.NettingSet("TREASURY-MULTICURRENCY", [swap, xccy], csa),
            qs.NettingSet("RATES-USD", [swap], csa_rates_desk),
        ],
    )
    xva_values = result.xva_values
    xva_sens = result.sensitivities
    exposures = result.exposures

xva_values

In [ ]:
# Compare CVA/FVA contributions across the client netting sets.
xva_plot = xva_values.pivot(index="measure", columns="netting_set", values="value").fillna(0.0)
ax = xva_plot.plot(kind="bar", figsize=(8, 4.5), width=0.75)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set(title="XVA by netting set", xlabel="Measure", ylabel="Value")
ax.legend(title="Netting set")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Expected positive/negative exposure profiles per netting set.
profile = exposures[0]
print(profile)
profile.to_dataframe().head(8)

In [ ]:
# EPE and ENE display the directional expected exposure over the simulation horizon.
fig, axes = plt.subplots(len(exposures), 1, figsize=(9, 3.5 * len(exposures)), sharex=True, squeeze=False)
for ax, exposure in zip(axes[:, 0], exposures):
    exposure_plot = exposure.to_dataframe().assign(date=lambda frame: pd.to_datetime(frame["date"]))
    ax.plot(exposure_plot["date"], exposure_plot["epe"], label="EPE", color="#2a9d8f")
    ax.plot(exposure_plot["date"], exposure_plot["ene"], label="ENE", color="#e76f51")
    ax.fill_between(exposure_plot["date"], 0, exposure_plot["ee"], color="#457b9d", alpha=0.18, label="EE")
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set(title=exposure.netting_set, ylabel="Expected exposure")
    ax.legend(ncols=3)
axes[-1, 0].set_xlabel("Date")
fig.suptitle("Expected exposure profiles", y=1.01)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# XVA sensitivities: market-quote deltas plus per-client credit/funding parameters.
xva_sens.reindex(xva_sens.value.abs().sort_values(ascending=False).index)